In [1]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score
)

ROOT = Path.cwd().parent
df = pd.read_csv(ROOT / "data" / "processed" / "telco_clean.csv")

In [2]:
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"].map({"No": 0, "Yes": 1})

X_train,X_temp,y_train,y_temp=train_test_split(
    X,y,test_size=0.40,random_state=42,stratify=y
)

X_val,X_test,y_val,y_test=train_test_split(
    X_temp,y_temp,test_size=0.50,random_state=42,stratify=y_temp
)

print(f"Train: {len(X_train):,} | Validation: {len(X_val):,} | Test: {len(X_test):,}")



Train: 4,225 | Validation: 1,409 | Test: 1,409


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [4]:
numeric_features=["tenure","MonthlyCharges","TotalCharges"]
categorical_features=[col for col in X.columns if col not in numeric_features]

numeric_pipeline=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

categorical_pipeline=Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])

preprocessor=ColumnTransformer([
    ("numeric",numeric_pipeline,numeric_features),
    ("categorical",categorical_pipeline,categorical_features)
])  


In [5]:
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("SeniorCitizen included:", "SeniorCitizen" in categorical_features)

Numeric features: 3
Categorical features: 16
SeniorCitizen included: True


In [6]:
models={
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight="balanced"
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        min_samples_leaf=5,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )
}

results=[]
trained_models={}

for name,classifier in models.items():
    pipeline=Pipeline([
        ("preprocessor",preprocessor),
        ("classifier",classifier)
    ])

    pipeline.fit(X_train,y_train)
    val_prob=pipeline.predict_proba(X_val)[:,1]
    val_pred=(val_prob>=0.5).astype(int)

    results.append({
        "Model": name,
        "ROC AUC": roc_auc_score(y_val,val_prob),
        "PR-AUC": average_precision_score(y_val,val_prob),
        "Precision": precision_score(y_val,val_pred),
        "Recall": recall_score(y_val,val_pred),
        "F1 Score": f1_score(y_val,val_pred)
    })

    trained_models[name]=pipeline

    results_df=pd.DataFrame(results).sort_values(
        "ROC AUC",ascending=False
    ).reset_index(drop=True)

    results_df.round(4)

In [7]:

best_model_name = results_df.loc[0, "Model"]
best_model = trained_models[best_model_name]

print("Best validation model:", best_model_name)
print(results_df.iloc[0].to_string())


Best validation model: Logistic Regression
Model        Logistic Regression
ROC AUC                 0.859451
PR-AUC                  0.678239
Precision               0.517647
Recall                  0.823529
F1 Score                0.635707


In [8]:
MODEL_PATH = ROOT / "models" / "best_model.joblib"
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, MODEL_PATH)

print("Selected model:", best_model_name)
print("Saved to:", MODEL_PATH)


Selected model: Logistic Regression
Saved to: d:\Instant AI\AI\Graduation Project\RetainAI\models\best_model.joblib
